In [26]:
from lingpy import *
import pandas as pd
from segments.tokenizer import Tokenizer

In [49]:
df = pd.read_csv('lingpy_input_unfiltered.tsv', sep='\t')

# Remove rows where 'form' is empty or only whitespace
df = df[df['form'].notna() & (df['form'].str.strip() != '')]
df = df[df['Language'] != 'ZAMBOANGUENO']

df.to_csv('lingpy_input_larger.tsv', sep='\t', index=False)

In [50]:
# Preparing the input data for LingPy

wl = Wordlist('lingpy_input_larger.tsv')
op = Tokenizer('lingpy_created_profile_unfiltered.tsv')

wl.add_entries('tokens', "form", op, column='IPA')
wl.output('tsv', filename='lingpy_foranalysis', ignore='all',
    prettify=False)
for idx, doculect, form, tokens in wl.iter_rows('doculect', 'form', 'tokens'):
    if form != tokens.replace(' ', ''):
        print('{0:10} {1:10} {2:15}'.format(doculect, form, tokens)) 

2025-07-10 14:57:54,920 [INFO] Data has been written to file <lingpy_foranalysis.tsv>.


CAIJIA     Sin        s i n          
CAIJIA     Sisu       s i s u        
CASIGURAN NEGRITO kEjo       k æ j o        
DUMAGAT CASIGURAN Esaʔ       es a ʔ         
DUPANINGAN AGTA tagilEtel  t a g i l æ t e l
DUPANINGAN AGTA nagEn      n a g æ n      
DUPANINGAN AGTA Esa        es a           
DUPANINGAN AGTA maʔEnta    m a ʔ æ n t a  
DUPANINGAN AGTA sEŋgit     s æ ŋ g i t    
ENGLISH    Ei         æ i            
ENGLISH    brEst      b r es t       
ENGLISH    dEi        d æ i          
ENGLISH    Ei         æ i            
ENGLISH    fEir       f æ i r        
ENGLISH    fiS        f i s          
ENGLISH    hEnd       h æ n d        
ENGLISH    nEit       n æ i t        
ENGLISH    pE8        p æ            
ENGLISH    tu8        t u            
FINNISH    minE       m i n æ        
FINNISH    silmE      s i l m æ      
FINNISH    tEjsi      t æ js i       
FINNISH    kEsi       k es i         
FINNISH    tEi        t æ i          
FINNISH    nenE       n e n æ        
FINNISH  

In [51]:
# Cleaning the output data
df = pd.read_csv('lingpy_foranalysis.tsv', sep='\t')
df.replace('', pd.NA, inplace=True)
df_clean = df.dropna()
df_clean.to_csv('lingpy_foranalysis_cleaned.tsv', sep='\t', index=False)

In [52]:
# Runnning Automatic Cognate Detection
lex = LexStat('lingpy_foranalysis_cleaned.tsv')
lex.get_scorer()
lex.cluster(method='lexstat', threshold=0.45, ref='cognates')
lex.output(
'tsv',
filename='WORDLIST_PH_Cognates',
subset=True,
prettify=False,
ignore='all'
)

CORRESPONDENCE CALCULATION:   0%|          | 0/11858.0 [00:00<?, ?it/s]2025-07-10 14:58:31,290 [INFO] Calculating alignments for pair AGTA / AGTA.
2025-07-10 14:58:31,299 [INFO] Calculating alignments for pair AGTA / AKLANON.
2025-07-10 14:58:31,310 [INFO] Calculating alignments for pair AGTA / ATTA PAMPLONA.
2025-07-10 14:58:31,320 [INFO] Calculating alignments for pair AGTA / BABUYAN.
2025-07-10 14:58:31,328 [INFO] Calculating alignments for pair AGTA / BALANGAW.
2025-07-10 14:58:31,339 [INFO] Calculating alignments for pair AGTA / BALANGINGI SAMA.
2025-07-10 14:58:31,348 [INFO] Calculating alignments for pair AGTA / BALANGINGI SAMA (1).
2025-07-10 14:58:31,358 [INFO] Calculating alignments for pair AGTA / BASQUE.
2025-07-10 14:58:31,369 [INFO] Calculating alignments for pair AGTA / BATAK PALAWAN.
2025-07-10 14:58:31,380 [INFO] Calculating alignments for pair AGTA / BILAAN KORONADAL.
CORRESPONDENCE CALCULATION:   0%|          | 11/11858.0 [00:00<01:48, 108.94it/s]2025-07-10 14:58:31,

In [53]:
from lingpy.compare.sanity import mutual_coverage_check, mutual_coverage_subset
for i in range(210, 0, -1):
    if mutual_coverage_check(wl, i):
        print("Minimal mutual coverage is at {0} concept pairs.".format(i))
        break

Minimal mutual coverage is at 20 concept pairs.


In [21]:
from lingpy.evaluate.acd import bcubes, diff

In [54]:
alm = Alignments(lex, ref='cognates')
alm.align()

2025-07-10 16:43:11,385 [WARNING] There are empty segments in the consensus.
2025-07-10 16:43:11,388 [INFO] 
	1 5 7
	1 0 7
2025-07-10 16:43:11,588 [WARNING] There are empty segments in the consensus.
2025-07-10 16:43:11,590 [INFO] 
	[0, 1, 0, 0, 1, 7, 0]
2025-07-10 16:43:12,372 [WARNING] There are empty segments in the consensus.
2025-07-10 16:43:12,375 [INFO] 
	4 7 1 7 4
	4 0 7 4
2025-07-10 16:43:12,436 [WARNING] There are empty segments in the consensus.
2025-07-10 16:43:12,438 [INFO] 
	1 0 7
	1 7 5 7


In [55]:
alm.output('html', filename="ALM_LTS_FTRD")

2025-07-10 16:43:32,300 [INFO] Data has been written to file <C:\Users\ramma\AppData\Local\Temp\tmpcrn78or_.alm>.
2025-07-10 16:43:33,705 [INFO] Data has been written to file <ALM_LTS_FTRD.html>.


In [56]:
# Converting to CSV
df = pd.read_csv('WORDLIST_PH_cognates.tsv', sep='\t')
df.to_csv('WORDLIST_PH_cognates_LexStat_small.csv', index=False)